# jlens-audit — kernel persistant

**Règle** : la cellule SETUP s'exécute UNE fois. Ne jamais redémarrer le kernel sans validation humaine. Chaque expérience écrit dans `results/` et `figs/`.

In [ ]:
# SETUP — une seule fois
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.load_model import load, layers, get_resid
tok, model = load()
print('modèle chargé —', model.config.num_hidden_layers, 'couches ; scannées :', layers())

In [ ]:
# LENSES — après avoir résolu les # ADAPTER de src/lens.py en lisant lenses/README.md
from src.lens import load_all
lenses = load_all()
print({k: len(v.maps) for k, v in lenses.items()})

## Étape 2 — go/no-go des lenses, puis conformité douce
**Go/no-go (binaire, décisif)** : `identity_check` (l'ancre `J_62 = I` doit reproduire le logit lens) + `orientation_check` (recouvrement à la sous-ancre : attrape une transposition, ce que l'ancre ne voit pas). Si l'un casse → STOP, le montage des lenses est faux.

**Conformité douce** (« sushi → Japan »), ordre de grandeur : R-lens dès les couches précoces, J-lens plus tard, logit lens tard ou jamais.

In [ ]:
from src.validate import smoke
smoke(lenses)

## Étape 3 — validation quantitative (test 1 du go/no-go)

In [ ]:
from src.validate import pass_at_k
pass_at_k(lenses)   # -> results/validation_multihop.json, figs/validation_multihop.png

## Étape 4 — capacité du modèle sur les paires pilote (test 2)

In [ ]:
from src import capability
capability.main('pairs_pilot.jsonl')   # cible >= 80 % de détection en clair par famille

## Étape 5 — test de temps sur une paire (test 3)

In [ ]:
import json, time
from src.scan import scan_text
p = json.loads(open('../data/pairs_pilot.jsonl').readline())
t0 = time.time(); out, toks = scan_text(p['anomalous'], lenses); print(f'{time.time()-t0:.1f}s pour {len(toks)} positions')
# regarder à la main quelques positions autour de l'anomalie :
from src.serialize import serialize
print('\n'.join(serialize(out['jlens']).split('\n')[:15]))

## Journal
Avant de fermer : entrée dans `experiments.md` (fait / vérifié / doute / prochaine étape).